# Tea Leaf Quality Prediction - Model Training
Trains two XGBoost models:
1. XGBClassifier → predicts quality class (poor / High / premium)
2. XGBRegressor → predicts percentage (grade score 88-100)

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, r2_score
from xgboost import XGBClassifier, XGBRegressor

In [ ]:
dataset_path = '../../Updated_Quality_Dataset.xlsx'
dataset_path = os.path.abspath(dataset_path)
print(f"Loading dataset from: {dataset_path}")
df = pd.read_excel(dataset_path)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
feature_cols = ['Tea Flavor', 'Base Price (Rs)', 'Moisture (%)', 
                'Quality Score', 'Caffeine (%)', 'Fineness (%)', 'Batch Weight (kg)']
target_class = 'quality'
target_reg = 'percentage'
X = df[feature_cols].copy()
y_class = df[target_class].copy()
y_reg = df[target_reg].copy()

In [ ]:
tea_flavor_encoder = LabelEncoder()
X['Tea Flavor'] = tea_flavor_encoder.fit_transform(X['Tea Flavor'])
quality_encoder = LabelEncoder()
y_class_encoded = quality_encoder.fit_transform(y_class)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_cls_train, y_cls_test, y_reg_train, y_reg_test = train_test_split(
    X_scaled, y_class_encoded, y_reg, test_size=0.2, random_state=42, stratify=y_class_encoded
)

In [ ]:
clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    use_label_encoder=False
)
clf.fit(X_train, y_cls_train)
y_cls_pred = clf.predict(X_test)
print(f"Classification Accuracy: {accuracy_score(y_cls_test, y_cls_pred):.4f}")
print(classification_report(y_cls_test, y_cls_pred, target_names=quality_encoder.classes_.tolist()))

In [ ]:
reg = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='rmse'
)
reg.fit(X_train, y_reg_train)
y_reg_pred = reg.predict(X_test)
print(f"Regression MAE: {mean_absolute_error(y_reg_test, y_reg_pred):.4f}")
print(f"Regression R2: {r2_score(y_reg_test, y_reg_pred):.4f}")

In [ ]:
save_dir = 'quality_models'
os.makedirs(save_dir, exist_ok=True)
clf.save_model(os.path.join(save_dir, 'quality_classifier.json'))
reg.save_model(os.path.join(save_dir, 'quality_regressor.json'))
with open(os.path.join(save_dir, 'tea_flavor_encoder.pkl'), 'wb') as f:
    pickle.dump(tea_flavor_encoder, f)
with open(os.path.join(save_dir, 'quality_encoder.pkl'), 'wb') as f:
    pickle.dump(quality_encoder, f)
with open(os.path.join(save_dir, 'feature_scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
metadata = {
    'feature_columns': feature_cols,
    'tea_flavors': tea_flavor_encoder.classes_.tolist(),
    'quality_classes': quality_encoder.classes_.tolist(),
    'classification_accuracy': float(accuracy_score(y_cls_test, y_cls_pred)),
    'regression_mae': float(mean_absolute_error(y_reg_test, y_reg_pred)),
    'regression_r2': float(r2_score(y_reg_test, y_reg_pred)),
    'training_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0])
}
with open(os.path.join(save_dir, 'model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)
print("Models saved successfully!")